In [0]:
%sql
CREATE TABLE IF NOT EXISTS employee_payments (
emp_id INT PRIMARY KEY,
emp_name VARCHAR(50),
department VARCHAR(30),
base_salary DECIMAL(10,2),
bonus DECIMAL(10,2),
joining_date DATE
);


INSERT INTO employee_payments VALUES
(1,'karthik','Data',75000.75,5000.50,'2019-03-15'),
(2,'veena','HR',65000.40,4000.25,'2021-06-20'),
(3,'ravi','Data',85000.90,6000.75,'2016-01-10'),
(4,'anil','Finance',70000.10,NULL,'2020-09-01'),
(5,'suresh','HR',60000.55,3000.30,'2022-11-25');

num_affected_rows,num_inserted_rows
5,5


In [0]:
%sql
select initcap(emp_name), sum(base_salary+coalesce(bonus,0)) as total_income, round(sum(base_salary+coalesce(bonus,0))) as rounded_income, year(joining_date)
from employee_payments
group by emp_name, joining_date

initcap(emp_name),total_income,rounded_income,year(joining_date)
Karthik,80001.25,80001,2019
Veena,69000.65,69001,2021
Ravi,91001.65,91002,2016
Anil,70000.10,70000,2020
Suresh,63000.85,63001,2022


In [0]:
%sql
ALTER TABLE employee_payments
ADD COLUMN joining_year INT;
UPDATE employee_payments
SET joining_year = EXTRACT(YEAR FROM joining_date);

num_affected_rows
5


In [0]:
%sql
SELECT *,(YEAR(CURRENT_DATE) - joining_year) AS experience,
CASE
    WHEN experience > 7 THEN 'Senior'
    WHEN experience BETWEEN 4 AND 7 THEN 'Mid'
    ELSE 'Junior'
END AS experience_level

FROM employee_payments;
     

emp_id,emp_name,department,base_salary,bonus,joining_date,joining_year,experience,experience_level
1,karthik,Data,75000.75,5000.50,2019-03-15,2019,7,Mid
2,veena,HR,65000.40,4000.25,2021-06-20,2021,5,Mid
3,ravi,Data,85000.90,6000.75,2016-01-10,2016,10,Senior
4,anil,Finance,70000.10,null,2020-09-01,2020,6,Mid
5,suresh,HR,60000.55,3000.30,2022-11-25,2022,4,Mid


In [0]:
%sql
CREATE TABLE IF NOT EXISTS orders_delivery (
order_id INT,
customer_name VARCHAR(50),
order_date DATE,
delivery_date DATE,
order_amount DECIMAL(10,2)
);
-- Insert Data
INSERT INTO orders_delivery VALUES
(101,'rajesh','2025-01-01','2025-01-05',12500.75),
(102,'meena','2025-01-10','2025-01-10',8400.40),
(103,'arun','2025-01-15','2025-01-20',15600.90),
(104,'pooja','2025-01-18',NULL,9200.10);
     

num_affected_rows,num_inserted_rows
4,4


In [0]:
%sql

SELECT 
    order_id,
    UPPER(customer_name) AS customer_name,
    order_date,
    -- Replace NULL with today
    COALESCE(delivery_date, CURRENT_DATE) AS delivery_date,
    -- Delivery days
    DATEDIFF(COALESCE(delivery_date, CURRENT_DATE), order_date) AS delivery_days,
    -- Truncate amount not available so using round
    ROUND(order_amount, 1) AS order_amount,
    CASE 
        WHEN delivery_date IS NULL THEN 'Pending'
        WHEN DATEDIFF(delivery_date, order_date) = 0 THEN 'Same-day'
        WHEN DATEDIFF(delivery_date, order_date) > 3 THEN 'Delayed'
        ELSE 'On-time'
    END AS delivery_status
FROM orders_delivery;

order_id,customer_name,order_date,delivery_date,delivery_days,order_amount,delivery_status
101,RAJESH,2025-01-01,2025-01-05,4,12500.8,Delayed
102,MEENA,2025-01-10,2025-01-10,0,8400.4,Same-day
103,ARUN,2025-01-15,2025-01-20,5,15600.9,Delayed
104,POOJA,2025-01-18,2026-04-10,447,9200.1,Pending


In [0]:
%sql
CREATE TABLE customer_spending (
cust_id INT,
cust_name VARCHAR(50),
city VARCHAR(30),
purchase_amount DECIMAL(10,2),
purchase_date DATE
);
--Insert Data
INSERT INTO customer_spending VALUES
(1,'amit','mumbai',12000.75,'2024-12-01'),
(2,'neha','delhi',8500.40,'2024-12-15'),
(3,'rohit','mumbai',15500.90,'2024-11-20'),
(4,'kavya','chennai',6000.10,'2024-10-05');

num_affected_rows,num_inserted_rows
4,4


In [0]:
%sql
SELECT 
    cust_id,
    INITCAP(cust_name) AS cust_name,
    MONTHNAME(purchase_date) AS purchase_month,
    ROUND(purchase_amount) AS rounded_amount,
    ABS(purchase_amount) AS absolute_amount,
    CASE 
        WHEN purchase_amount > 15000 THEN 'High'
        WHEN purchase_amount BETWEEN 8000 AND 15000 THEN 'Medium'
        ELSE 'Low'
    END AS spending_category
FROM customer_spending;

cust_id,cust_name,purchase_month,rounded_amount,absolute_amount,spending_category
1,Amit,Dec,12001,12000.75,Medium
2,Neha,Dec,8500,8500.40,Medium
3,Rohit,Nov,15501,15500.90,High
4,Kavya,Oct,6000,6000.10,Low
